# Vintern GGUF Invoice OCR Local Pipeline

Notebook này là pipeline tự chứa: input là ảnh hoặc PDF, PDF sẽ được render thành ảnh, model `rootonchair/Vintern-1B-v3_5-GGUF-ext` đọc hóa đơn qua `llama.cpp server`, output là file JSON theo cấu trúc demo.

Vintern là vision-language model nên `llama-server` cần cả model GGUF và file `mmproj`. Notebook sẽ tự tải `mmproj`, sau đó in ra command server để bạn copy sang terminal riêng.

```powershell
llama-server -hf rootonchair/Vintern-1B-v3_5-GGUF-ext:Q4_K_M --mmproj "D:/Invoice_Scanner/models/mmproj-Vintern-1B-v3_5-Q8_0.gguf" --chat-template vicuna --port 8081
```

Nếu muốn nhẹ hơn có thể đổi `Q4_K_M` thành `Q4_K_S` hoặc `IQ4_XS`.

In [ ]:
import sys

!{sys.executable} -m pip install --upgrade pillow pymupdf requests huggingface_hub

In [ ]:
import ast
import base64
import json
import re
from io import BytesIO
from pathlib import Path

import pymupdf as fitz
import requests
from huggingface_hub import hf_hub_download
from IPython.display import display
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

## Cấu hình input/output

Chỉ cần đổi `INPUT_PATH`. Có thể là `.pdf`, `.jpg`, `.png`, `.webp`, `.tif`, ...

In [ ]:
INPUT_PATH = Path("D:/Invoice_Scanner/4f30c1ad-c130-47a0-b164-41ad070218f2.pdf")

SERVER_URL = "http://127.0.0.1:8081/v1"
SERVER_MODEL = "local-model"

MODEL_REPO_ID = "rootonchair/Vintern-1B-v3_5-GGUF-ext"
MODEL_QUANT = "Q4_K_M"
MMPROJ_REPO_ID = "ngxson/Vintern-1B-v3_5-GGUF"
MMPROJ_FILENAME = "mmproj-Vintern-1B-v3_5-Q8_0.gguf"
MODEL_DIR = Path("D:/Invoice_Scanner/models")
MMPROJ_PATH = MODEL_DIR / MMPROJ_FILENAME

PDF_DPI = 300
MAX_IMAGE_SIDE = 2600
JPEG_QUALITY = 95
MAX_TOKENS = 2048
TEMPERATURE = 0.0
USE_IMAGE_ENHANCEMENT = True
USE_PDF_TEXT_LAYER = True
INCLUDE_RAW_RESPONSE_IN_RESULTS = False

OUTPUT_JSON = INPUT_PATH.with_suffix(".vintern_gguf.json")
PDF_IMAGE_DIR = INPUT_PATH.with_name(f"{INPUT_PATH.stem}_images")

print("Input:", INPUT_PATH)
print("Output JSON:", OUTPUT_JSON)
print("Server:", SERVER_URL)
print("MMProj:", MMPROJ_PATH)

## Tải mmproj và mở server

Chạy cell này trước, copy command được in ra sang một terminal PowerShell riêng, rồi giữ terminal server đó mở.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

downloaded_mmproj = Path(hf_hub_download(
    repo_id=MMPROJ_REPO_ID,
    filename=MMPROJ_FILENAME,
    local_dir=MODEL_DIR,
))

MMPROJ_PATH = downloaded_mmproj.resolve()

if not MMPROJ_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy mmproj sau khi tải: {MMPROJ_PATH}")

server_command = (
    f'llama-server -hf {MODEL_REPO_ID}:{MODEL_QUANT} '
    f'--mmproj "{MMPROJ_PATH}" '
    f'--chat-template vicuna '
    f'--port 8081'
)

print("Đã có mmproj:", MMPROJ_PATH)
print("Kiểm tra PowerShell:", f'Test-Path "{MMPROJ_PATH}"')
print("\nCopy lệnh này sang terminal PowerShell riêng và giữ terminal đó mở:\n")
print(server_command)

In [ ]:
INVOICE_JSON_SCHEMA = {
    "invoice": {
        "invoice_type": "VAT_INVOICE | SALES_INVOICE | RECEIPT | OTHER | null",
        "invoice_number": "string | null",
        "series": "string | null",
        "invoice_date": "YYYY-MM-DD | null",
        "tax_authority_code": "string | null",
        "currency": "string | null",
        "payment_method": "string | null",
    },
    "seller": {
        "name": "string | null",
        "english_name": "string | null",
        "tax_code": "string | null",
        "address": "string | null",
        "phone": ["string"],
        "fax": ["string"],
        "email": "string | null",
        "website": "string | null",
    },
    "buyer": {
        "name": "string | null",
        "company_name": "string | null",
        "tax_code": "string | null",
        "address": "string | null",
        "id_card": "string | null",
        "passport_number": "string | null",
        "account_number": "string | null",
        "budgetary_unit_code": "string | null",
    },
    "shipping": {
        "ship_from_warehouse": "string | null",
    },
    "items": [
        {
            "line_number": "number | null",
            "description": "string | null",
            "unit": "string | null",
            "quantity": "number | null",
            "unit_price": "number | null",
            "amount": "number | null",
        }
    ],
    "totals": {
        "subtotal": "number | null",
        "vat_rate": "number | null",
        "vat_amount": "number | null",
        "total_payment": "number | null",
        "amount_in_words": "string | null",
    },
    "signature": {
        "is_valid": "boolean | null",
        "signed_by": "string | null",
        "signed_date": "YYYY-MM-DD | null",
    },
    "metadata": {
        "source_type": "PDF | IMAGE | null",
        "page_count": "number | null",
        "ocr_processed": True,
        "pdf_text_layer_used": "boolean | null",
    },
}

SYSTEM_MESSAGE = "Bạn là một mô hình trí tuệ nhân tạo đa phương thức Tiếng Việt có tên gọi là Vintern, được phát triển bởi người Việt. Bạn là một trợ lý trí tuệ nhân tạo hữu ích và không gây hại."
MTMD_MEDIA_MARKER = "<__media__>"
VINTERN_IMAGE_MARKER = "<image>"

INVOICE_PROMPT = f"""
Bạn là hệ thống OCR và trích xuất thông tin từ hóa đơn Việt Nam.

Trả về duy nhất một JSON object hợp lệ, không dùng markdown, không giải thích.

Quy tắc bắt buộc:
- JSON root phải có đúng các khóa cấp 1: invoice, seller, buyer, shipping, items, totals, signature, metadata.
- Không trả JSON phẳng kiểu seller_name, buyer_tax_code, invoice_date ở cấp root.
- Mọi thông tin người bán phải nằm trong seller; người mua trong buyer; tổng tiền trong totals.
- Nếu không thấy trường nào thì để null. Nếu không có danh sách thì dùng [].
- Chuẩn hóa ngày về YYYY-MM-DD nếu có thể.
- Với số tiền, số lượng, đơn giá: trả về number, bỏ dấu phân cách hàng nghìn.
- Giữ nguyên tiếng Việt có dấu trong tên công ty, địa chỉ và mô tả hàng hóa.
- Không tự bịa thông tin không có trên ảnh.

Schema bắt buộc:
{json.dumps(INVOICE_JSON_SCHEMA, ensure_ascii=False, indent=2)}
""".strip()

In [ ]:
IMAGE_EXTENSIONS = {
    ".bmp", ".gif", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"
}


def pdf_to_images(pdf_path, dpi=200):
    document = fitz.open(pdf_path)
    images = []
    matrix = fitz.Matrix(dpi / 72, dpi / 72)

    for page_index in range(len(document)):
        page = document.load_page(page_index)
        pixmap = page.get_pixmap(matrix=matrix, alpha=False)
        image = Image.frombytes("RGB", (pixmap.width, pixmap.height), pixmap.samples)
        images.append(image)

    document.close()
    return images


def save_images_to_files(images, output_dir, stem, image_format="jpg", jpeg_quality=95):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    extension = image_format.lower().lstrip(".")
    pil_format = "JPEG" if extension in {"jpg", "jpeg"} else extension.upper()
    extension = "jpg" if extension == "jpeg" else extension
    padding = max(2, len(str(len(images))))
    paths = []

    for page_index, image in enumerate(images, start=1):
        page_image = ImageOps.exif_transpose(image)
        if pil_format == "JPEG":
            page_image = page_image.convert("RGB")

        image_path = output_dir / f"{stem}_page_{page_index:0{padding}d}.{extension}"

        if pil_format == "JPEG":
            page_image.save(image_path, format=pil_format, quality=jpeg_quality, optimize=True)
        else:
            page_image.save(image_path, format=pil_format)

        paths.append(image_path)

    return paths


def load_input_images(input_path, pdf_dpi=200, save_pdf_images=True, pdf_image_dir=None):
    input_path = Path(input_path)
    suffix = input_path.suffix.lower()

    if suffix == ".pdf":
        images = pdf_to_images(input_path, dpi=pdf_dpi)
        image_dir = Path(pdf_image_dir) if pdf_image_dir else input_path.with_name(f"{input_path.stem}_images")
        image_paths = save_images_to_files(images, image_dir, input_path.stem) if save_pdf_images else [None] * len(images)
        return images, "PDF", image_paths

    if suffix in IMAGE_EXTENSIONS:
        image = ImageOps.exif_transpose(Image.open(input_path)).convert("RGB")
        return [image], "IMAGE", [input_path]

    raise ValueError(f"Input không hỗ trợ: {input_path}. Hãy dùng PDF hoặc ảnh.")


def enhance_invoice_image(image):
    image = ImageOps.autocontrast(image, cutoff=1)
    image = ImageEnhance.Contrast(image).enhance(1.15)
    image = ImageEnhance.Sharpness(image).enhance(1.25)
    return image.filter(ImageFilter.UnsharpMask(radius=1.2, percent=120, threshold=3))


def normalize_image(image, max_side=1800):
    image = ImageOps.exif_transpose(image).convert("RGB")
    image.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    if USE_IMAGE_ENHANCEMENT:
        image = enhance_invoice_image(image)
    return image


def image_to_data_uri(image, max_side=1800, jpeg_quality=92):
    image = normalize_image(image, max_side=max_side)
    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=jpeg_quality, optimize=True)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{encoded}"


def extract_json_text(text):
    if not isinstance(text, str):
        text = str(text)

    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    if start == -1:
        return text

    depth = 0
    in_string = False
    escape = False

    for index in range(start, len(text)):
        character = text[index]

        if escape:
            escape = False
            continue

        if character == "\\":
            escape = True
            continue

        if character == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if character == "{":
            depth += 1
        elif character == "}":
            depth -= 1
            if depth == 0:
                return text[start:index + 1]

    return text[start:]


def parse_model_json(response):
    json_text = extract_json_text(response)

    try:
        parsed = json.loads(json_text)
    except json.JSONDecodeError as first_error:
        try:
            parsed = ast.literal_eval(json_text)
        except Exception:
            return {"parse_error": str(first_error), "raw_model_response": response}

    if isinstance(parsed, dict):
        return parsed

    return {"parse_error": "Model response is valid JSON but not an object.", "raw_model_response": response}


def blank_invoice_json():
    return {
        "invoice": {
            "invoice_type": None,
            "invoice_number": None,
            "series": None,
            "invoice_date": None,
            "tax_authority_code": None,
            "currency": None,
            "payment_method": None,
        },
        "seller": {
            "name": None,
            "english_name": None,
            "tax_code": None,
            "address": None,
            "phone": [],
            "fax": [],
            "email": None,
            "website": None,
        },
        "buyer": {
            "name": None,
            "company_name": None,
            "tax_code": None,
            "address": None,
            "id_card": None,
            "passport_number": None,
            "account_number": None,
            "budgetary_unit_code": None,
        },
        "shipping": {
            "ship_from_warehouse": None,
        },
        "items": [],
        "totals": {
            "subtotal": None,
            "vat_rate": None,
            "vat_amount": None,
            "total_payment": None,
            "amount_in_words": None,
        },
        "signature": {
            "is_valid": None,
            "signed_by": None,
            "signed_date": None,
        },
        "metadata": {
            "source_type": None,
            "page_count": None,
            "ocr_processed": True,
            "pdf_text_layer_used": False,
        },
    }


FLAT_FIELD_MAP = {
    "type": ("invoice", "invoice_type"),
    "invoice_type": ("invoice", "invoice_type"),
    "number": ("invoice", "invoice_number"),
    "invoice_number": ("invoice", "invoice_number"),
    "series": ("invoice", "series"),
    "date": ("invoice", "invoice_date"),
    "invoice_date": ("invoice", "invoice_date"),
    "tax_authority_code": ("invoice", "tax_authority_code"),
    "currency": ("invoice", "currency"),
    "payment_method": ("invoice", "payment_method"),
    "tax_code": ("seller", "tax_code"),
    "address": ("seller", "address"),
    "phone": ("seller", "phone"),
    "fax": ("seller", "fax"),
    "email": ("seller", "email"),
    "website": ("seller", "website"),
    "seller_name": ("seller", "name"),
    "seller_english_name": ("seller", "english_name"),
    "seller_tax_code": ("seller", "tax_code"),
    "seller_address": ("seller", "address"),
    "seller_phone": ("seller", "phone"),
    "seller_fax": ("seller", "fax"),
    "seller_email": ("seller", "email"),
    "seller_website": ("seller", "website"),
    "buyer_name": ("buyer", "name"),
    "buyer_company_name": ("buyer", "company_name"),
    "buyer_tax_code": ("buyer", "tax_code"),
    "buyer_address": ("buyer", "address"),
    "buyer_id_card": ("buyer", "id_card"),
    "buyer_passport_number": ("buyer", "passport_number"),
    "buyer Passport_number": ("buyer", "passport_number"),
    "buyer_account_number": ("buyer", "account_number"),
    "buyer_budget_unit_code": ("buyer", "budgetary_unit_code"),
    "ship_from_warehouse": ("shipping", "ship_from_warehouse"),
    "subtotal": ("totals", "subtotal"),
    "vat_rate": ("totals", "vat_rate"),
    "vat_amount": ("totals", "vat_amount"),
    "total_payment": ("totals", "total_payment"),
    "amount_in_words": ("totals", "amount_in_words"),
    "signature_is_valid": ("signature", "is_valid"),
    "signed_by": ("signature", "signed_by"),
    "signed_date": ("signature", "signed_date"),
}


SECTION_FIELD_ALIASES = {
    "invoice": {
        "type": "invoice_type",
        "number": "invoice_number",
        "date": "invoice_date",
    }
}


def normalize_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]


def normalize_date_value(value):
    if not isinstance(value, str):
        return value

    match = re.fullmatch(r"\s*(\d{1,2})[/-](\d{1,2})[/-](\d{4})\s*", value)
    if not match:
        return value

    day, month, year = match.groups()
    return f"{year}-{int(month):02d}-{int(day):02d}"


def apply_section_aliases(section, value):
    if not isinstance(value, dict):
        return value

    value = dict(value)
    for alias, canonical in SECTION_FIELD_ALIASES.get(section, {}).items():
        if alias in value and canonical not in value:
            value[canonical] = value[alias]
    return value


def merge_known_fields(target, source):
    for key, value in source.items():
        if key not in target or value is None:
            continue
        if key.endswith("date"):
            value = normalize_date_value(value)
        if isinstance(target[key], dict) and isinstance(value, dict):
            merge_known_fields(target[key], value)
        elif key in {"phone", "fax"}:
            target[key] = normalize_list(value)
        elif key == "items" and isinstance(value, list):
            target[key] = [normalize_item(item) for item in value if isinstance(item, dict)]
        else:
            target[key] = value


def normalize_item(item):
    template = {
        "line_number": None,
        "description": None,
        "unit": None,
        "quantity": None,
        "unit_price": None,
        "amount": None,
    }
    merge_known_fields(template, item)
    return template


def normalize_invoice_data(data):
    if not isinstance(data, dict) or "parse_error" in data:
        return data

    normalized = blank_invoice_json()

    for section in ["invoice", "seller", "buyer", "shipping", "totals", "signature", "metadata"]:
        value = data.get(section)
        if isinstance(value, list) and value and isinstance(value[0], dict):
            value = value[0]
        value = apply_section_aliases(section, value)
        if isinstance(value, dict):
            merge_known_fields(normalized[section], value)

    if isinstance(data.get("items"), list):
        normalized["items"] = [normalize_item(item) for item in data["items"] if isinstance(item, dict)]

    for flat_key, (section, field) in FLAT_FIELD_MAP.items():
        if flat_key in data and data[flat_key] is not None:
            value = data[flat_key]
            if field.endswith("date"):
                value = normalize_date_value(value)
            if field in {"phone", "fax"}:
                value = normalize_list(value)
            normalized[section][field] = value

    return normalized


def clean_text_value(value):
    if value is None:
        return None
    value = re.sub(r"[ \t]+", " ", str(value))
    value = re.sub(r"\s*\n\s*", " ", value)
    value = value.strip(" :-")
    return value or None


def regex_value(pattern, text, flags=re.IGNORECASE | re.DOTALL, group=1):
    match = re.search(pattern, text, flags)
    if not match:
        return None
    return clean_text_value(match.group(group))


def normalize_tax_code(value):
    value = clean_text_value(value)
    if not value:
        return None
    digits = re.sub(r"\D", "", value)
    return digits or value


def parse_vn_number(value):
    if value is None or isinstance(value, (int, float)):
        return value

    text = re.sub(r"[^0-9,\.\-]", "", str(value).strip())
    if not text:
        return None

    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    elif text.count(".") > 1:
        text = text.replace(".", "")
    elif "." in text:
        left, right = text.split(".", 1)
        if len(right) == 3:
            text = left + right

    try:
        number = float(text)
    except ValueError:
        return value

    return int(number) if number.is_integer() else number


def parse_vn_percent(value):
    value = clean_text_value(value)
    if not value:
        return None
    return parse_vn_number(value.replace("%", ""))


def extract_pdf_text_pages(pdf_path):
    document = fitz.open(pdf_path)
    pages = [page.get_text("text") for page in document]
    document.close()
    return pages


def parse_invoice_items_from_text(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    start_index = None
    end_index = len(lines)
    for index, line in enumerate(lines):
        if "6 = 4 x 5" in line:
            start_index = index + 1
            break

    if start_index is None:
        return []

    for index in range(start_index, len(lines)):
        if "Cộng tiền hàng" in lines[index] or "Total amount" in lines[index]:
            end_index = index
            break

    table_lines = lines[start_index:end_index]
    items = []
    index = 0
    while index + 5 < len(table_lines):
        if not re.fullmatch(r"\d+", table_lines[index]):
            index += 1
            continue

        unit = table_lines[index + 2]
        if not re.fullmatch(r"[A-Za-zÀ-ỹ/%\.]+", unit):
            index += 1
            continue

        items.append({
            "line_number": int(table_lines[index]),
            "description": clean_text_value(table_lines[index + 1]),
            "unit": clean_text_value(unit),
            "quantity": parse_vn_number(table_lines[index + 3]),
            "unit_price": parse_vn_number(table_lines[index + 4]),
            "amount": parse_vn_number(table_lines[index + 5]),
        })
        index += 6

    return items


def parse_invoice_text_layer(text):
    data = blank_invoice_json()
    if not clean_text_value(text):
        return data

    data["invoice"]["invoice_type"] = "VAT_INVOICE" if re.search(r"H[ÓO]A ĐƠN GI[ÁA] TRỊ GIA TĂNG|VAT INVOICE", text, re.IGNORECASE) else None
    data["invoice"]["invoice_number"] = regex_value(r"Số \(No\):[ \t]*([^\n]+)", text)
    data["invoice"]["series"] = regex_value(r"Ký hiệu \(Series\):[ \t]*([^\n]+)", text)
    day = regex_value(r"Ngày \(Date\)[ \t]*(\d{1,2})", text)
    month = regex_value(r"Tháng \(Month\)[ \t]*(\d{1,2})", text)
    year = regex_value(r"Năm \(Year\)[ \t]*(\d{4})", text)
    if day and month and year:
        data["invoice"]["invoice_date"] = f"{year}-{int(month):02d}-{int(day):02d}"
    data["invoice"]["tax_authority_code"] = regex_value(r"Mã CQT:[ \t]*([^\n]*)", text, flags=re.IGNORECASE)
    data["invoice"]["payment_method"] = regex_value(r"Hình thức thanh toán \(Method payment\):[ \t]*([^\n]+)", text, flags=re.IGNORECASE)

    seller_match = re.search(
        r"(?m)^([^\n:]+)\n([^\n:]+)\nMã số thuế \(Tax code\):\s*([^\n]+)\nĐịa chỉ \(Address\):\s*(.*?)\nĐT \(Tel\):\s*(.*?)\nEmail:\s*(.*?)\s+Website:\s*([^\n]+)",
        text,
        re.IGNORECASE | re.DOTALL,
    )
    if seller_match:
        data["seller"].update({
            "name": clean_text_value(seller_match.group(1)),
            "english_name": clean_text_value(seller_match.group(2)),
            "tax_code": normalize_tax_code(seller_match.group(3)),
            "address": clean_text_value(seller_match.group(4)),
            "email": clean_text_value(seller_match.group(6)),
            "website": clean_text_value(seller_match.group(7)),
        })
        contact_line = clean_text_value(seller_match.group(5)) or ""
        phone_part = contact_line.split("Fax:", 1)[0]
        fax_part = contact_line.split("Fax:", 1)[1] if "Fax:" in contact_line else ""
        data["seller"]["phone"] = [item.strip() for item in re.split(r"\s+-\s+", phone_part) if item.strip()]
        data["seller"]["fax"] = [item.strip() for item in re.split(r"\s+-\s+", fax_part) if item.strip()]

    data["buyer"]["name"] = regex_value(r"Họ tên người mua hàng \(Buyer's name\):[ \t]*([^\n]*)", text, flags=re.IGNORECASE)
    data["buyer"]["company_name"] = regex_value(r"Tên đơn vị \(Company's name\):\s*(.*?)\n\s*Mã số thuế", text)
    data["buyer"]["tax_code"] = normalize_tax_code(regex_value(r"Tên đơn vị \(Company's name\):.*?\n\s*Mã số thuế \(Tax code\):\s*([^\n]+)", text))
    data["buyer"]["address"] = regex_value(r"Địa chỉ:\s*\(Address\):\s*(.*?)\n\s*Hình thức thanh toán", text)
    data["buyer"]["id_card"] = regex_value(r"CCCD \(ID Card\):[ \t]*([^\n]*)", text, flags=re.IGNORECASE)
    data["buyer"]["account_number"] = regex_value(r"Số tài khoản \(Account number\):[ \t]*([^\n]*)", text, flags=re.IGNORECASE)
    data["buyer"]["budgetary_unit_code"] = regex_value(r"Mã ĐVQHNS \(Budgetary Unit Code\):[ \t]*([^\n]*)", text, flags=re.IGNORECASE)
    data["buyer"]["passport_number"] = regex_value(r"Số hộ chiếu \(PP Number\):[ \t]*([^\n]*)", text, flags=re.IGNORECASE)

    data["shipping"]["ship_from_warehouse"] = regex_value(r"Kho xuất hàng \(Ship-from warehouse\):\s*(.*?)\n\s*Mã ĐVQHNS", text)
    data["items"] = parse_invoice_items_from_text(text)

    data["totals"]["subtotal"] = parse_vn_number(regex_value(r"Cộng tiền hàng \(Total amount\):\s*([^\n]+)", text))
    data["totals"]["vat_rate"] = parse_vn_percent(regex_value(r"Thuế suất GTGT \(VAT rate\):\s*([^\n]+)", text))
    data["totals"]["vat_amount"] = parse_vn_number(regex_value(r"Tiền thuế GTGT \(VAT amount\):\s*([^\n]+)", text))
    data["totals"]["total_payment"] = parse_vn_number(regex_value(r"Tổng cộng tiền thanh toán \(Total payment\):\s*([^\n]+)", text))
    data["totals"]["amount_in_words"] = regex_value(r"Số tiền viết bằng chữ \(Amount in words\):\s*([^\n]+)", text)

    data["signature"]["is_valid"] = True if re.search(r"Signature Valid", text, re.IGNORECASE) else None
    data["signature"]["signed_by"] = regex_value(r"Ký bởi:\s*([^\n]+)", text)
    data["signature"]["signed_date"] = regex_value(r"Ngày ký:[ \t]*([^\n]*)", text, flags=re.IGNORECASE)
    data["metadata"]["pdf_text_layer_used"] = True

    return normalize_invoice_data(data)


def has_meaningful_value(value):
    if value is None or value == "" or value == []:
        return False
    if isinstance(value, dict):
        return any(has_meaningful_value(item) for item in value.values())
    return True


TEXT_LAYER_CAN_CLEAR = {
    ("invoice", "tax_authority_code"),
    ("buyer", "name"),
    ("buyer", "id_card"),
    ("buyer", "passport_number"),
    ("buyer", "account_number"),
    ("buyer", "budgetary_unit_code"),
    ("signature", "signed_date"),
}


def overlay_invoice_data(base, overlay):
    base = normalize_invoice_data(base)
    overlay = normalize_invoice_data(overlay)
    if not isinstance(base, dict) or "parse_error" in base:
        base = blank_invoice_json()

    for section, overlay_value in overlay.items():
        if section == "items":
            if overlay_value:
                base["items"] = overlay_value
            continue

        if isinstance(overlay_value, dict) and isinstance(base.get(section), dict):
            for key, value in overlay_value.items():
                if has_meaningful_value(value) or (section, key) in TEXT_LAYER_CAN_CLEAR:
                    base[section][key] = value

    return base


In [ ]:
def server_origin(server_url):
    server_url = server_url.rstrip("/")
    return server_url[:-3].rstrip("/") if server_url.endswith("/v1") else server_url


def server_chat_urls(server_url):
    origin = server_origin(server_url)
    base = server_url.rstrip("/")
    if not base.endswith("/v1"):
        base = f"{base}/v1"
    return list(dict.fromkeys([f"{base}/chat/completions", f"{origin}/chat/completions"]))


def probe_server(server_url, timeout=10):
    origin = server_origin(server_url)
    urls = [origin, f"{origin}/v1/models", f"{origin}/props", f"{origin}/health"]
    rows = []

    for url in urls:
        try:
            response = requests.get(url, timeout=timeout)
            rows.append({
                "url": url,
                "status_code": response.status_code,
                "content_type": response.headers.get("content-type", ""),
                "body": response.text[:300],
            })
        except requests.RequestException as error:
            rows.append({"url": url, "error": str(error)})

    return rows


def get_server_props(server_url, timeout=10):
    url = f"{server_origin(server_url)}/props"
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()


def get_server_media_marker(server_url):
    props = get_server_props(server_url)
    return props.get("media_marker") or MTMD_MEDIA_MARKER


def response_to_text(response_json):
    if isinstance(response_json, dict):
        content = response_json.get("content")
        if isinstance(content, str):
            return content

        completion = response_json.get("completion")
        if isinstance(completion, str):
            return completion

    choices = response_json.get("choices") if isinstance(response_json, dict) else None
    if choices:
        message = choices[0].get("message", {})
        content = message.get("content")
        if isinstance(content, str):
            return content
    return str(response_json)


def image_to_base64_jpeg(image, max_side=1800, jpeg_quality=92):
    image = normalize_image(image, max_side=max_side)
    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=jpeg_quality, optimize=True)
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def strip_media_markers(prompt):
    prompt = prompt or INVOICE_PROMPT
    prompt = re.sub(r"<__media[^>]*__>\s*", "", prompt)
    prompt = prompt.replace(VINTERN_IMAGE_MARKER, "")
    return prompt.strip()


def build_completion_prompt(prompt, server_url=SERVER_URL):
    media_marker = get_server_media_marker(server_url)
    clean_prompt = strip_media_markers(prompt)
    return (
        f"<|im_start|>system\n{SYSTEM_MESSAGE}<|im_end|>"
        f"<|im_start|>user\n{media_marker}\n{VINTERN_IMAGE_MARKER}\n{clean_prompt}<|im_end|>"
        f"<|im_start|>assistant\n"
    )


def completion_url(server_url):
    return f"{server_origin(server_url)}/completion"


def run_vintern_server(image, prompt=None, server_url=SERVER_URL, model=SERVER_MODEL, max_tokens=2048, temperature=0.0, max_image_side=1800, jpeg_quality=92, timeout=600):
    prompt = build_completion_prompt(prompt, server_url=server_url)
    image_base64 = image_to_base64_jpeg(image, max_side=max_image_side, jpeg_quality=jpeg_quality)
    payload = {
        "prompt": {
            "prompt_string": prompt,
            "multimodal_data": [image_base64],
        },
        "n_predict": max_tokens,
        "temperature": temperature,
        "stream": False,
        "cache_prompt": False,
        "ignore_eos": True,
        "stop": ["<|im_end|>", "<|endoftext|>", "</s>", "\n\n*"],
        "repeat_penalty": 1.05,
    }

    url = completion_url(server_url)

    try:
        response = requests.post(url, json=payload, timeout=timeout)
    except requests.RequestException as error:
        raise RuntimeError(
            "Không kết nối được llama.cpp server. Hãy chạy cell 'Tải mmproj và mở server', "
            "copy command được in ra sang terminal PowerShell riêng."
        ) from error

    if not response.ok:
        error_text = response.text[:1000]
        if "mmproj" in error_text.lower() or "image input is not supported" in error_text.lower():
            raise RuntimeError(
                "Server chưa load multimodal projector (mmproj), nên chưa đọc được ảnh.\n"
                "Hãy dừng terminal llama-server cũ, chạy lại command được in ở cell 'Tải mmproj và mở server', rồi OCR lại.\n\n"
                f"Chi tiết server:\nHTTP {response.status_code}\n{error_text}"
            )

        if "media markers" in error_text.lower() or "failed to tokenize prompt" in error_text.lower():
            media_marker = get_server_media_marker(server_url)
            raise RuntimeError(
                "Server vẫn báo lỗi media marker.\n"
                f"Media marker server đang dùng: {media_marker}\n"
                "Hãy chạy lại cell định nghĩa INVOICE_PROMPT và cell định nghĩa hàm, rồi OCR lại.\n\n"
                f"Chi tiết server:\nHTTP {response.status_code}\n{error_text}"
            )

        raise RuntimeError(f"Server lỗi tại {url}: HTTP {response.status_code}\n{error_text}")

    response_json = response.json()
    output_text = response_to_text(response_json)

    if not output_text.strip():
        raise RuntimeError(
            "Server trả về nội dung rỗng. Có thể prompt format chưa khớp model hoặc model dừng ngay.\n"
            "Notebook đã đổi sang template gốc Vintern '<|im_start|>user/assistant'. Hãy chạy lại cell định nghĩa hàm rồi OCR lại.\n\n"
            f"Prompt đã gửi bắt đầu bằng:\n{prompt[:500]}\n\n"
            f"Full response JSON:\n{json.dumps(response_json, ensure_ascii=False, indent=2)[:2000]}"
        )

    return output_text


def scan_invoice_file(input_path, server_url=SERVER_URL, model=SERVER_MODEL, include_raw_response=INCLUDE_RAW_RESPONSE_IN_RESULTS):
    images, source_type, image_paths = load_input_images(
        input_path,
        pdf_dpi=PDF_DPI,
        save_pdf_images=Path(input_path).suffix.lower() == ".pdf",
        pdf_image_dir=PDF_IMAGE_DIR,
    )
    pdf_text_pages = []
    if USE_PDF_TEXT_LAYER and Path(input_path).suffix.lower() == ".pdf":
        pdf_text_pages = extract_pdf_text_pages(input_path)

    results = []
    for page_index, image in enumerate(images, start=1):
        print(f"Đang OCR trang {page_index}/{len(images)}...")
        raw_response = run_vintern_server(
            image=image,
            server_url=server_url,
            model=model,
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            max_image_side=MAX_IMAGE_SIDE,
            jpeg_quality=JPEG_QUALITY,
        )
        page_data = normalize_invoice_data(parse_model_json(raw_response))

        if page_index - 1 < len(pdf_text_pages) and clean_text_value(pdf_text_pages[page_index - 1]):
            text_layer_data = parse_invoice_text_layer(pdf_text_pages[page_index - 1])
            page_data = overlay_invoice_data(page_data, text_layer_data)

        if not isinstance(page_data.get("metadata"), dict):
            page_data["metadata"] = {}

        page_data["metadata"].update({
            "source_type": source_type,
            "page_count": len(images),
            "ocr_processed": True,
        })

        if image_paths[page_index - 1]:
            page_data["metadata"]["page_image_path"] = str(image_paths[page_index - 1])

        result_row = {
            "page": page_index,
            "data": page_data,
        }
        if include_raw_response:
            result_row["raw_response"] = raw_response
        results.append(result_row)

    return results


def save_results(results, output_path, include_raw_response=False):
    output = []
    for row in results:
        row = dict(row)
        if not include_raw_response:
            row.pop("raw_response", None)
        output.append(row)

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(output, file, ensure_ascii=False, indent=2)


## Kiểm tra server

In [ ]:
for item in probe_server(SERVER_URL):
    print(item)

print("Media marker server:", get_server_media_marker(SERVER_URL))

## Xem trước input

In [ ]:
preview_images, preview_source_type, preview_image_paths = load_input_images(
    INPUT_PATH,
    pdf_dpi=PDF_DPI,
    save_pdf_images=INPUT_PATH.suffix.lower() == ".pdf",
    pdf_image_dir=PDF_IMAGE_DIR,
)

print("Loại file:", preview_source_type)
print("Số trang/ảnh:", len(preview_images))
if any(preview_image_paths):
    print("Ảnh đã lưu:", preview_image_paths[0])

display(preview_images[0])

## OCR và lưu JSON

In [ ]:
results = scan_invoice_file(INPUT_PATH, server_url=SERVER_URL, model=SERVER_MODEL)
results

In [ ]:
save_results(results, OUTPUT_JSON, include_raw_response=False)
print("Đã lưu:", OUTPUT_JSON)